# Convergence project — real-data figures for the viva (final v6)

## Purpose

This notebook produces **three figures computed directly from the raw cached embedding vectors** — not replotted from a results table. They are the visual evidence behind three claims in the report:

1. **Independently trained encoders arrange items the same way** — visible as item-level pairing in a 2-D projection.
2. **Shape agreement varies by pair type** — text↔text > image siblings > cross-modal > GPT-2, with bge–SBERT topping at ρ=0.768. Matches Fig A15b to the decimal.
3. **The 67% mean neighbourhood overlap is not an artifact of averaging** — the per-item histogram shows the distribution.

## What makes these figures different from the report's tables

Every number in the report was measured by a notebook and quoted in prose or a table. These figures go one step further: they are **computed live from the raw vectors** on your L4, producing images an examiner can trace back to the cached embeddings rather than to a number someone typed. When asked "did you just replot your own tables?" — these are the answer.

## Data source

All encoders are loaded from **`hub_rebuilt.npz`** (7 hub members, 9,533 items, row-aligned). **ConvNeXt-base** (the held-out encoder, never in the hub) is loaded separately from `e1_img_ckpt_convnext-base-224-22k_native.npz`. If found, all figures expand to include it (8 encoders, 28 pairs); if not, they fall back to 7 encoders (21 pairs).

## How the scatter pairs are chosen (auto-pick)

Figure 2 (the heatmap) computes **all** pairwise ρ values first. Figure 1 then picks the three most informative panels automatically:
- **Best text ↔ text pair** — expected: bge-m3 ↔ SBERT (shared objective, no shared lineage).
- **Best DINOv2 ↔ text pair** — whichever DINOv2 variant (small/base/large) has the highest ρ with any text encoder.
- **Best ConvNeXt ↔ text pair** — the cross-modal pair we have NEVER measured until now. ConvNeXt differs from every text encoder on two axes (architecture: convnet; supervision: 22k class labels), so this is the hardest test of raw geometric agreement.

The contrast across the three panels IS the local-structure finding: pairing is tight when shape agreement is high, and looser across modalities — but still present.

## Run order

1. **Cell 1** — mount Drive.
2. **Cell 2** — load all encoders, print roster.
3. **Cell 3** — compute Figure 2 (heatmap, all pairs), then Figure 1 (auto-picked scatters), then Figure 3 (kNN histogram).

All figures display inline AND save to `figs_for_claude/` on Drive.

In [ ]:
# Cell 1 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — load all encoder vectors
#
# hub_rebuilt.npz holds the 7 hub members' RAW vectors (before any hub or map),
# each (9533, d_k), row-aligned by item. ConvNeXt is loaded separately because
# it was deliberately held out of the hub to serve as an independent test case.
# If ConvNeXt has a different row count (different cache run), all encoders are
# truncated to the shorter count so every figure compares the SAME items.

import os, numpy as np
import matplotlib
import matplotlib.pyplot as plt

FOLDER = "/content/drive/MyDrive/convergence_experiment"
OUT    = os.path.join(FOLDER, "figs_for_claude")
os.makedirs(OUT, exist_ok=True)

NAVY = "#172B54"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8.5,
                     "text.color": NAVY, "axes.labelcolor": NAVY,
                     "xtick.color": NAVY, "ytick.color": NAVY,
                     "axes.titlesize": 9.5, "axes.titleweight": "bold"})

def l2n(V):
    """L2-normalise each row to unit length."""
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

# --- 7 hub encoders ---
z = np.load(os.path.join(FOLDER, "hub_rebuilt.npz"), allow_pickle=True)
names = [str(x) for x in z["encoder_names"]]
raw   = {n: np.asarray(z[f"raw_{n}"]) for n in names}
N     = raw[names[0]].shape[0]
print("Hub encoders:", names, "| N =", N)

# --- ConvNeXt: the held-out encoder ---
print("\n=== loading ConvNeXt ===")
convnext_vec = None
CONVNEXT_FILE = "e1_img_ckpt_convnext-base-224-22k_native.npz"
convnext_path = os.path.join(FOLDER, CONVNEXT_FILE)
if os.path.exists(convnext_path):
    zc = np.load(convnext_path, allow_pickle=True)
    print(f"  {CONVNEXT_FILE} keys: {zc.files}")
    for k in zc.files:
        a = np.asarray(zc[k])
        print(f"    {k}: shape={a.shape} dtype={a.dtype}")
        if a.ndim == 2 and a.shape[1] >= 512:
            convnext_vec = a
            print(f"  -> USING key '{k}'")
            break
else:
    print(f"  {CONVNEXT_FILE} not found in {FOLDER}")

if convnext_vec is not None:
    Nc = convnext_vec.shape[0]
    if Nc == N:
        raw["convnext"] = convnext_vec
    elif Nc > N:
        raw["convnext"] = convnext_vec[:N]
        print(f"  truncated ConvNeXt from {Nc} to {N} rows")
    else:
        print(f"  ConvNeXt has {Nc} rows vs hub {N} - truncating ALL to {Nc}")
        raw = {n: raw[n][:Nc] for n in raw}
        raw["convnext"] = convnext_vec
        N = Nc
    names = names + ["convnext"]
    print(f"\nConvNeXt added: shape {raw['convnext'].shape}")
else:
    print("\nConvNeXt NOT loaded. Figures will use 7 encoders only.")

# --- summary ---
print(f"\nTotal encoders: {len(names)}, N = {N}")
for n in names:
    print(f"    {n:12s} shape={raw[n].shape}")

# --- full readable names for every label, legend, and axis tick ---
LAB = {
    "img_small": "DINOv2-small", "img_base": "DINOv2-base", "img_large": "DINOv2-large",
    "txt_bge": "bge-m3", "txt_gpt2": "GPT-2", "txt_bert": "BERT", "txt_sbert": "SBERT",
    "convnext": "ConvNeXt-base",
}
def lab(n): return LAB.get(n, n)

# --- encoder group sets (used for auto-picking scatter pairs) ---
IMG_DINO = {"img_small", "img_base", "img_large"}
IMG_ALL  = IMG_DINO | {"convnext"}
TXT_ALL  = set(names) - IMG_ALL

In [ ]:
# Cell 3 — compute all three figures
#
# ORDER MATTERS: Figure 2 (heatmap) runs FIRST because it computes all pairwise
# rho values. Figure 1 (scatter) then reads from that matrix to auto-pick the
# three most informative panels.

from scipy.spatial.distance import pdist
from scipy.stats import rankdata

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FIGURE 2 — shape-agreement heatmap (computed first, used by Fig 1) ║
# ╚══════════════════════════════════════════════════════════════════════╝
# For each encoder, compute pairwise cosine distances over ALL N items (45.4M
# distances per encoder at N=9533). To fit in memory, each distance vector is
# converted to unit-normalized float32 RANKS — Spearman then reduces to a single
# dot product, exact to ~1e-9 vs scipy.stats.spearmanr.
#
# Why not just call spearmanr directly? Memory: holding 8 raw distance vectors
# in float64 costs 2.9 GB; the rank trick costs 1.45 GB as float32 and makes
# each of the 28 correlations a one-line dot product rather than a re-ranking.

print("=== Figure 2: shape-agreement heatmap ===")
try:
    ranks = {}
    for n_ in names:
        d = pdist(l2n(raw[n_]), metric="cosine")  # 45.4M float64, transient
        r = rankdata(d).astype(np.float64)         # exact ranks in float64
        del d                                       # free the 0.36 GB distance vector
        r -= r.mean()                               # center
        r /= np.linalg.norm(r)                      # unit-normalise: now in [-1,1]
        ranks[n_] = r.astype(np.float32)            # safe as float32 (small magnitudes)
        print(f"    ranked {lab(n_)}")

    K = len(names)
    Mx = np.eye(K)
    for i in range(K):
        for j in range(i+1, K):
            rho = float(np.dot(ranks[names[i]], ranks[names[j]]))  # = Spearman
            Mx[i, j] = Mx[j, i] = rho

    # --- plot ---
    fig, ax = plt.subplots(figsize=(6.2, 5.6) if K > 7 else (5.8, 5.2), dpi=200)
    im = ax.imshow(Mx, cmap="RdYlBu_r", vmin=0, vmax=1)
    labels = [lab(n) for n in names]
    ax.set_xticks(range(K)); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7.5)
    ax.set_yticks(range(K)); ax.set_yticklabels(labels, fontsize=7.5)
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{Mx[i,j]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if (Mx[i,j] > 0.6 or Mx[i,j] < 0.2) else "black")
    n_pairs = K * (K - 1) // 2
    title_suffix = " (+ ConvNeXt held-out)" if K > 7 else " (= Fig A15b)"
    ax.set_title("Shape agreement - Spearman of pairwise distances\n"
                 "%d pairs, no map fitted, all %d items%s" % (n_pairs, N, title_suffix))
    fig.colorbar(im, ax=ax, fraction=0.046).set_label("rank correlation rho")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "real_shape_agreement_21pairs.png"))
    plt.show()
    plt.close(fig)
    np.savez(os.path.join(OUT, "shape_agreement_matrix.npz"), names=names, M=Mx, n_items=N)
    del ranks

    # --- print summary ---
    order = sorted([(Mx[i,j], names[i], names[j])
                     for i in range(K) for j in range(i+1,K)], reverse=True)
    cm = [Mx[i,j] for i in range(K) for j in range(i+1,K)
          if (names[i] in IMG_ALL) != (names[j] in IMG_ALL)]
    print("[2] shape agreement heatmap OK  (%d pairs, all %d items)" % (n_pairs, N))
    print("    top pair:    %s - %s  rho=%.3f" % (lab(order[0][1]), lab(order[0][2]), order[0][0]))
    print("    bottom pair: %s - %s  rho=%.3f" % (lab(order[-1][1]), lab(order[-1][2]), order[-1][0]))
    print("    cross-modal mean rho = %.3f" % (sum(cm)/len(cm)))
    if "convnext" in names:
        ci = names.index("convnext")
        print("\n    ConvNeXt-base rho against each encoder:")
        for j in range(K):
            if j != ci:
                print(f"      {lab(names[j]):16s}  rho = {Mx[ci,j]:.3f}")
except Exception as e:
    print("[2] skipped:", e)
    Mx = None

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FIGURE 1 — three paired scatters, auto-picked from the heatmap    ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Each panel shows the same 300 items as dots in two encoders' spaces,
# joined by colour. If two encoders agree on where items go, same-coloured
# dots land near each other (short tie-lines). If they disagree, the lines
# cross the plot.
#
# Because different encoders have different widths (768 to 2048), a joint
# PCA is undefined. Instead, each encoder is PCA'd to 2-D separately, then
# the two 2-D frames are PROCRUSTES-ALIGNED (best rotation, same items as
# anchors). This is the honest way to compare frames of different dimension.
#
# The three panels are auto-picked from Figure 2's rho matrix:
#   1. Best text <-> text pair    (expected: bge-m3 <-> SBERT, rho ~0.77)
#   2. Best DINOv2 <-> text pair  (expected: DINOv2-large <-> BERT, rho ~0.42)
#   3. Best ConvNeXt <-> text pair (NEVER MEASURED BEFORE — this is the new one)
#
# The contrast across the three panels is the local-structure finding itself.

print("\n=== Figure 1: paired scatters (auto-picked) ===")
try:
    def paired_scatter(ax, a_name, b_name, rho, n=300, seed=0):
        """Draw one scatter panel: two encoders, same items, joined by colour."""
        A, B = l2n(raw[a_name]), l2n(raw[b_name])
        idx = np.random.default_rng(seed).choice(min(len(A), len(B)), n, replace=False)
        A, B = A[idx], B[idx]
        def pca2(X):
            Xc = X - X.mean(0)
            U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
            return Xc @ Vt[:2].T
        PA, PB = pca2(A), pca2(B)
        # Procrustes: rotate B's 2-D frame onto A's (minimise sum of squared distances)
        M = PA.T @ PB
        U, _, Vt = np.linalg.svd(M)
        PB = PB @ (Vt.T @ U.T)
        colors = plt.cm.turbo(np.linspace(0, 1, n))
        for i in range(n):
            ax.plot([PA[i,0], PB[i,0]], [PA[i,1], PB[i,1]],
                    color=colors[i], lw=0.3, alpha=0.35)
        ax.scatter(PA[:,0], PA[:,1], c=colors, marker="o", s=13,
                   edgecolor="none", label=lab(a_name))
        ax.scatter(PB[:,0], PB[:,1], c=colors, marker="^", s=15,
                   edgecolor="none", label=lab(b_name))
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        ax.legend(frameon=False, fontsize=7, loc="upper right")
        ax.set_title(f"{lab(a_name)}  <->  {lab(b_name)}\nrho = {rho:.3f}", fontsize=9.5)
        ax.set_aspect("equal", adjustable="datalim")

    def best_pair(group_a, group_b):
        """Find the pair (a in group_a, b in group_b) with the highest rho."""
        best = (-1, None, None)
        for i in range(K):
            for j in range(i+1, K):
                a, b = names[i], names[j]
                if (a in group_a and b in group_b) or (b in group_a and a in group_b):
                    if Mx[i,j] > best[0]:
                        # return in (group_a_member, group_b_member) order
                        if a in group_a:
                            best = (Mx[i,j], a, b)
                        else:
                            best = (Mx[i,j], b, a)
        return best  # (rho, name_a, name_b)

    if Mx is not None:
        # auto-pick the three panels
        rho_tt, a_tt, b_tt = best_pair(TXT_ALL, TXT_ALL)
        rho_dt, a_dt, b_dt = best_pair(IMG_DINO, TXT_ALL)
        print(f"    best text-text:    {lab(a_tt)} <-> {lab(b_tt)}  rho={rho_tt:.3f}")
        print(f"    best DINOv2-text:  {lab(a_dt)} <-> {lab(b_dt)}  rho={rho_dt:.3f}")

        if "convnext" in names:
            rho_ct, a_ct, b_ct = best_pair({"convnext"}, TXT_ALL)
            print(f"    best ConvNeXt-text:{lab(a_ct)} <-> {lab(b_ct)}  rho={rho_ct:.3f}")
            fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14.4, 4.8), dpi=200)
            paired_scatter(ax1, a_tt, b_tt, rho_tt)
            paired_scatter(ax2, a_dt, b_dt, rho_dt)
            paired_scatter(ax3, a_ct, b_ct, rho_ct)
            fig.suptitle(
                "Same items, joined by colour: pairing strength tracks shape agreement\n"
                f"{lab(a_tt)} <-> {lab(b_tt)} (rho={rho_tt:.2f})  |  "
                f"{lab(a_dt)} <-> {lab(b_dt)} (rho={rho_dt:.2f})  |  "
                f"{lab(a_ct)} <-> {lab(b_ct)} (rho={rho_ct:.2f})",
                fontsize=10.5, fontweight="bold", y=1.02)
        else:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 5.0), dpi=200)
            paired_scatter(ax1, a_tt, b_tt, rho_tt)
            paired_scatter(ax2, a_dt, b_dt, rho_dt)
            fig.suptitle(
                "Same items, joined by colour: pairing is visible when shape agreement is high\n"
                f"{lab(a_tt)} <-> {lab(b_tt)} (rho={rho_tt:.2f})  |  "
                f"{lab(a_dt)} <-> {lab(b_dt)} (rho={rho_dt:.2f})",
                fontsize=11, fontweight="bold", y=1.02)

        fig.tight_layout()
        fig.savefig(os.path.join(OUT, "real_pca_paired_two.png"), bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("[1] real_pca_paired_two.png  OK")
    else:
        print("[1] skipped: heatmap (Figure 2) did not run, cannot auto-pick pairs")
except Exception as e:
    print("[1] skipped:", e)

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FIGURE 3 — per-item kNN-overlap histogram                         ║
# ╚══════════════════════════════════════════════════════════════════════╝
# For each item, count how many of its k nearest neighbours (by cosine) are
# the SAME items in both spaces. The report quotes the mean (67.1% at k=10
# for the B pair); this histogram shows the full DISTRIBUTION behind that mean.
#
# A mean of 67% could be everyone at 6-7/10 (uniform agreement) or half at
# 10/10 and half at 3/10 (bimodal). The histogram settles it.
#
# Uses DINOv2-base vs SBERT (a cross-modal pair) on a 2,000-item sample.
# Chance = k/m (dotted line). Full 9,533-item version would need a 700 MB
# similarity matrix — the sample is sufficient for the distribution shape.

print("\n=== Figure 3: kNN-overlap histogram ===")
try:
    m = 2000
    sel = np.random.default_rng(0).choice(N, m, replace=False)
    A, B = l2n(raw["img_base"])[sel], l2n(raw["txt_sbert"])[sel]
    k = 10
    SA = A @ A.T; np.fill_diagonal(SA, -np.inf)
    SB = B @ B.T; np.fill_diagonal(SB, -np.inf)
    nnA = np.argsort(-SA, 1)[:, :k]
    nnB = np.argsort(-SB, 1)[:, :k]
    ov = np.array([len(set(nnA[i]) & set(nnB[i])) / k for i in range(m)])
    fig, ax = plt.subplots(figsize=(5.4, 3.4), dpi=200)
    ax.hist(ov, bins=np.linspace(0, 1, 11), color="#2B5FD9", edgecolor="white")
    ax.axvline(ov.mean(), color="#B02418", ls="--", lw=1.4)
    ax.text(ov.mean()+0.02, ax.get_ylim()[1]*0.9,
            f"mean {ov.mean():.3f}", color="#B02418", fontsize=9)
    ax.axvline(k/m, color="#6B7280", ls=":", lw=1.2)
    ax.set_xlabel(f"fraction of {k} nearest neighbours shared")
    ax.set_ylabel("items")
    ax.set_title("Per-item neighbour overlap, DINOv2-base <-> SBERT (k=10)\n"
                 "distribution behind the mean-overlap headline")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "real_knn_overlap_hist.png"))
    plt.show()
    plt.close(fig)
    print(f"[3] real_knn_overlap_hist.png  OK  (mean {ov.mean():.3f} vs chance {k/m:.4f}, {m}-item sample)")
except Exception as e:
    print("[3] skipped:", e)

print("\nDone. All outputs in:", OUT)
print("Tell Claude when the PNGs are in figs_for_claude/.")